# 02-02_統計グラフ作成
## 目的
- CSVファイル毎に統計グラフを作成し、画像データとして保存する
- 統計グラフはトラヒック量と廃棄量の2軸

## CSVデータ
- 元のCSVのヘッダー情報は以下
  - タイムスタンプ
  - 回線番号
  - 下りのトラヒック量(Byte)
  - 上りのトラヒック量(Byte)
  - 下りの廃棄量(Packets)
  - 下りの廃棄量(Byte)
  - 下りトラヒック(Mbps)
  - 下り廃棄量(Mbps)
  - 下り合計(Mbps)

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob
import os

# --- 設定 ---
input_dir = "output"        # 加工済みCSVが入っているフォルダ
save_dir = "output"         # グラフ画像を保存するフォルダ

# 保存用フォルダがなければ作成
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"📁 フォルダを作成しました: {save_dir}")

# 1. 加工済みCSVの一覧を取得
file_list = sorted(glob.glob(os.path.join(input_dir, "*.csv")))

if not file_list:
    print(f"❌ エラー: {input_dir} 内にデータが見つかりません。")
else:
    print(f"📊 合計 {len(file_list)} 件のグラフを作成します...")

    # 2. 各ファイルを読み込んで描画
    for i, file_path in enumerate(file_list):
        pipe_df = pd.read_csv(file_path)
        pipe_df['タイムスタンプ'] = pd.to_datetime(pipe_df['タイムスタンプ'])
        
        # 回線番号を取得
        pipe_name = pipe_df['回線番号'].iloc[0]
        safe_name = os.path.basename(file_path).replace(".csv", "")

        fig, ax1 = plt.subplots(figsize=(12, 6))
        ax2 = ax1.twinx()

        # 下り合計(Mbps): ピンク
        ax1.plot(pipe_df['タイムスタンプ'], pipe_df['下り合計(Mbps)'], 
                 color='hotpink', label='Total (Traffic + Drop) [Mbps]', linewidth=2, zorder=2)
        
        # 下りトラヒック(Mbps): 青
        ax1.plot(pipe_df['タイムスタンプ'], pipe_df['下りトラヒック(Mbps)'], 
                 color='blue', label='Downstream Traffic [Mbps]', linewidth=1.5, zorder=3)
        
        # 下り廃棄量(Packets): 緑半透明（第2軸）
        ax2.bar(pipe_df['タイムスタンプ'], pipe_df['下りの廃棄量(Packets)'], 
                color='green', alpha=0.3, label='Drop [Packets]', width=0.003, zorder=1)

        # 軸・タイトルの設定
        ax1.set_title(f'Traffic Analysis - {pipe_name}', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Throughput (Mbps)')
        ax2.set_ylabel('Drop Count (Packets)')
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        
        # 凡例・グリッド
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax1.legend(h1 + h2, l1 + l2, loc='upper left', frameon=True, shadow=True)
        ax1.grid(True, linestyle=':', alpha=0.6)
        
        plt.tight_layout()

        # --- 画像として保存 ---
        save_path = os.path.join(save_dir, f"{safe_name}.png")
        plt.savefig(save_path, dpi=150) # 高画質で保存
        
        # ノートブック上の表示を維持（不要なら plt.close(fig) にするとメモリ節約）
        plt.show()
        
        if (i + 1) % 5 == 0:
            print(f"処理済み: {i + 1} / {len(file_list)}")

    print(f"✨ すべてのグラフを '{save_dir}' に保存完了しました。")